In [3]:
import os

print(os.getcwd())

c:\Users\apran\OneDrive\Desktop\Olist_Customer_Experience_Analytics\notebooks


In [4]:
import pandas as pd

# Load orders and reviews
orders = pd.read_csv("../data/olist_orders_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# Convert date columns
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

# Create delivery delay
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

# Create late delivery flag
orders["late_delivery_flag"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

# Add review score
review_summary = (
    reviews.groupby("order_id")
    .agg(review_score=("review_score", "mean"))
    .reset_index()
)

orders = orders.merge(
    review_summary,
    on="order_id",
    how="left"
)

print("Phase 5 data loaded successfully.")
print("Orders:", orders.shape)
print("Reviews:", reviews.shape)

Phase 5 data loaded successfully.
Orders: (99441, 11)
Reviews: (99224, 7)


In [5]:
# Create delivery delay groups
orders["delivery_delay_group"] = pd.cut(
    orders["delivery_delay_days"],
    bins=[-float("inf"), -7, 0, 7, 30, float("inf")],
    labels=[
        "More than 7 days early",
        "1–7 days early",
        "On time",
        "1–7 days late",
        "More than 7 days late"
    ]
)

# Compare review scores
delay_review = (
    orders.groupby("delivery_delay_group", observed=True)
    .agg(
        avg_review_score=("review_score", "mean"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

delay_review["avg_review_score"] = (
    delay_review["avg_review_score"].round(2)
)

print(delay_review)

     delivery_delay_group  avg_review_score  order_count
0  More than 7 days early              4.32        71308
1          1–7 days early              4.20        17341
2                 On time              3.18         4481
3           1–7 days late              1.70         2986
4   More than 7 days late              2.02          360


In [7]:
# Load order items
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

# Calculate total order value
price_summary = (
    order_items.groupby("order_id")
    .agg(total_price=("price", "sum"))
    .reset_index()
)

# Add total price to orders
orders = orders.merge(
    price_summary,
    on="order_id",
    how="left"
)

print(orders[
    ["order_id", "total_price"]
].head(10))

                           order_id  total_price
0  e481f51cbdc54678b7cc49136f2d6af7        29.99
1  53cdb2fc8bc7dce0b6741e2150273451       118.70
2  47770eb9100c2d0c44946d9cf07ec65d       159.90
3  949d5b44dbf5de918fe9c16f97b45f8a        45.00
4  ad21c59c0840e6cb83a9ceb5573f8159        19.90
5  a4591c265e18cb1dcee52889e2d8acc3       147.90
6  136cce7faa42fdb2cefd53fdc79a6098        49.90
7  6514b8ad8028c9f2cc2374ded245783f        59.99
8  76c6e866289321a7c93b82b54852dc33        19.90
9  e69bfb5eb88e0ed6a785585b27e16dbf       149.99


In [8]:
orders["order_value_group"] = pd.cut(
    orders["total_price"],
    bins=[0, 50, 100, 250, 500, float("inf")],
    labels=[
        "Under 50",
        "50–100",
        "100–250",
        "250–500",
        "500+"
    ]
)

order_value_review = (
    orders.groupby("order_value_group", observed=True)
    .agg(
        avg_review_score=("review_score", "mean"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

order_value_review["avg_review_score"] = (
    order_value_review["avg_review_score"].round(2)
)

print(order_value_review)

  order_value_group  avg_review_score  order_count
0          Under 50              4.17        29654
1            50–100              4.12        28189
2           100–250              4.08        30016
3           250–500              3.97         7190
4              500+              3.94         3617


In [10]:
# Load order items
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

# Calculate total price and total freight for each order
freight_summary = (
    order_items.groupby("order_id")
    .agg(
        total_price_items=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate freight ratio
freight_summary["freight_ratio"] = (
    freight_summary["total_freight"]
    / freight_summary["total_price_items"]
)

# Add freight ratio to orders
orders = orders.merge(
    freight_summary[["order_id", "freight_ratio"]],
    on="order_id",
    how="left"
)

print(orders[
    ["order_id", "freight_ratio"]
].head(10))

                           order_id  freight_ratio
0  e481f51cbdc54678b7cc49136f2d6af7       0.290764
1  53cdb2fc8bc7dce0b6741e2150273451       0.191744
2  47770eb9100c2d0c44946d9cf07ec65d       0.120200
3  949d5b44dbf5de918fe9c16f97b45f8a       0.604444
4  ad21c59c0840e6cb83a9ceb5573f8159       0.438191
5  a4591c265e18cb1dcee52889e2d8acc3       0.184990
6  136cce7faa42fdb2cefd53fdc79a6098       0.321643
7  6514b8ad8028c9f2cc2374ded245783f       0.252875
8  76c6e866289321a7c93b82b54852dc33       0.806533
9  e69bfb5eb88e0ed6a785585b27e16dbf       0.131809


In [11]:
orders["freight_ratio_group"] = pd.cut(
    orders["freight_ratio"],
    bins=[0, 0.05, 0.10, 0.20, 0.50, float("inf")],
    labels=[
        "0–5%",
        "5–10%",
        "10–20%",
        "20–50%",
        "50%+"
    ]
)

freight_review = (
    orders.groupby("freight_ratio_group", observed=True)
    .agg(
        avg_review_score=("review_score", "mean"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

freight_review["avg_review_score"] = (
    freight_review["avg_review_score"].round(2)
)

print(freight_review)

  freight_ratio_group  avg_review_score  order_count
0                0–5%              4.09         3865
1               5–10%              4.15        11502
2              10–20%              4.13        28306
3              20–50%              4.10        39045
4                50%+              4.06        15610


In [12]:
# Load products and order items
products = pd.read_csv("../data/olist_products_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
translation = pd.read_csv("../data/product_category_name_translation.csv")

# Translate product categories to English
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products["product_category_final"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
)

# Connect orders → products → categories
order_categories = order_items[
    ["order_id", "product_id"]
].drop_duplicates()

order_categories = order_categories.merge(
    products[["product_id", "product_category_final"]],
    on="product_id",
    how="left"
)

# One category per order for this analysis
order_categories = (
    order_categories.groupby("order_id")
    .agg(product_category=("product_category_final", "first"))
    .reset_index()
)

orders = orders.merge(
    order_categories,
    on="order_id",
    how="left"
)

print(orders[
    ["order_id", "product_category", "review_score"]
].head(10))

                           order_id  product_category  review_score
0  e481f51cbdc54678b7cc49136f2d6af7        housewares           4.0
1  53cdb2fc8bc7dce0b6741e2150273451         perfumery           4.0
2  47770eb9100c2d0c44946d9cf07ec65d              auto           5.0
3  949d5b44dbf5de918fe9c16f97b45f8a          pet_shop           5.0
4  ad21c59c0840e6cb83a9ceb5573f8159        stationery           5.0
5  a4591c265e18cb1dcee52889e2d8acc3              auto           4.0
6  136cce7faa42fdb2cefd53fdc79a6098               NaN           2.0
7  6514b8ad8028c9f2cc2374ded245783f              auto           5.0
8  76c6e866289321a7c93b82b54852dc33   furniture_decor           1.0
9  e69bfb5eb88e0ed6a785585b27e16dbf  office_furniture           5.0


In [13]:
category_review = (
    orders.dropna(subset=["product_category", "review_score"])
    .groupby("product_category")
    .agg(
        avg_review_score=("review_score", "mean"),
        order_count=("order_id", "count")
    )
    .reset_index()
)

# Keep categories with enough orders for a reliable comparison
category_review = category_review[
    category_review["order_count"] >= 20
].copy()

category_review["avg_review_score"] = (
    category_review["avg_review_score"].round(2)
)

# Sort from lowest to highest
category_review = category_review.sort_values(
    "avg_review_score"
)

print("Lowest-rated categories:")
print(category_review.head(10))

print("\nHighest-rated categories:")
print(category_review.tail(10).sort_values(
    "avg_review_score",
    ascending=False
))

Lowest-rated categories:
                     product_category  avg_review_score  order_count
57                   office_furniture              3.62         1255
30              fashion_male_clothing              3.70          111
27             fashio_female_clothing              3.73           39
23                diapers_and_hygiene              3.74           27
41  furniture_mattress_and_upholstery              3.82           38
4                               audio              3.84          343
47                       home_confort              3.88          373
19          construction_tools_safety              3.89          161
46                     home_comfort_2              3.90           21
34                    fixed_telephony              3.90          214

Highest-rated categories:
                         product_category  avg_review_score  order_count
8                  books_general_interest              4.47          505
22                costruction_tools_tools  

In [14]:
# Load seller information from order items
seller_orders = order_items[
    ["order_id", "seller_id"]
].drop_duplicates()

# Connect seller information with order-level metrics
seller_data = orders.merge(
    seller_orders,
    on="order_id",
    how="inner"
)

# Calculate seller performance metrics
seller_performance = (
    seller_data.groupby("seller_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        on_time_rate=("late_delivery_flag", lambda x: (x == 0).mean() * 100),
        order_volume=("order_id", "nunique")
    )
    .reset_index()
)

# Normalize review score
seller_performance["review_score_normalized"] = (
    seller_performance["avg_review_score"] / 5
) * 100

# Calculate seller performance score
seller_performance["seller_performance_score"] = (
    0.5 * seller_performance["review_score_normalized"] +
    0.3 * seller_performance["on_time_rate"] +
    0.2 * (
        seller_performance["order_volume"]
        / seller_performance["order_volume"].max()
    ) * 100
)

print(seller_performance.head(10))

                          seller_id  avg_review_score  on_time_rate  \
0  0015a82c2db000af6aaaf3ae2ecb0532          3.666667    100.000000   
1  001cca7ae9ae17fb1caed9dfb1094831          3.984772     93.500000   
2  001e6ad469a905060d959994f1b41e4f          1.000000    100.000000   
3  002100f778ceb8431b7a1020ff7ab48f          3.901961     82.352941   
4  003554e2dce176b5555353e4f3555ac8          5.000000    100.000000   
5  004c9cd9d87a3c30c522c48c4fc07416          4.148387     91.772152   
6  00720abe85ba0859807595bbf045a33b          3.615385     84.615385   
7  00ab3eff1b5192e5f1a63bcecfee11c8          5.000000    100.000000   
8  00d8b143d12632bad99c0ad66ad52825          5.000000    100.000000   
9  00ee68308b45bc5e2660cd833c3f81cc          4.298507     91.111111   

   order_volume  review_score_normalized  seller_performance_score  
0             3                73.333333                 66.699029  
1           200                79.695431                 70.055213  
2          

In [15]:
# Create seller performance groups
seller_performance["performance_group"] = pd.cut(
    seller_performance["seller_performance_score"],
    bins=[0, 50, 60, 70, 80, 100],
    labels=[
        "0–50",
        "50–60",
        "60–70",
        "70–80",
        "80–100"
    ]
)

seller_review = (
    seller_performance
    .groupby("performance_group", observed=True)
    .agg(
        avg_review_score=("avg_review_score", "mean"),
        seller_count=("seller_id", "count")
    )
    .reset_index()
)

seller_review["avg_review_score"] = (
    seller_review["avg_review_score"].round(2)
)

print(seller_review)


  performance_group  avg_review_score  seller_count
0              0–50              1.52           228
1             50–60              2.99           233
2             60–70              3.81           893
3             70–80              4.38          1240
4            80–100              4.98           496


In [16]:
# Inspect customer review comments

print(reviews.columns)

print("\nSample review comments:")
print(reviews[[
    "review_score",
    "review_comment_message"
]].dropna().head(10))

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

Sample review comments:
    review_score                             review_comment_message
3              5              Recebi bem antes do prazo estipulado.
4              5  Parabéns lojas lannister adorei comprar pela I...
9              4  aparelho eficiente. no site a marca do aparelh...
12             4    Mas um pouco ,travando...pelo valor ta Boa.\r\n
15             5  Vendedor confiável, produto ok e entrega antes...
16             2  GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...
19             1                                            Péssimo
22             5                                       Loja nota 10
24             5              obrigado pela atençao amim dispensada
27             5  A compra foi realizada facilmente.\r\nA entreg...


In [17]:
# Prepare customer reviews for AI analysis

review_ai = reviews[
    ["review_id", "order_id", "review_score", "review_comment_message"]
].copy()

# Remove reviews without comments
review_ai = review_ai.dropna(
    subset=["review_comment_message"]
).copy()

# Clean text
review_ai["review_text"] = (
    review_ai["review_comment_message"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove empty comments
review_ai = review_ai[
    review_ai["review_text"] != ""
].copy()

print("Reviews available for AI analysis:", len(review_ai))

print("\nSample:")
print(
    review_ai[
        ["review_score", "review_text"]
    ].head(10)
)

Reviews available for AI analysis: 40950

Sample:
    review_score                                        review_text
3              5              Recebi bem antes do prazo estipulado.
4              5  Parabéns lojas lannister adorei comprar pela I...
9              4  aparelho eficiente. no site a marca do aparelh...
12             4        Mas um pouco ,travando...pelo valor ta Boa.
15             5  Vendedor confiável, produto ok e entrega antes...
16             2  GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...
19             1                                            Péssimo
22             5                                       Loja nota 10
24             5              obrigado pela atençao amim dispensada
27             5  A compra foi realizada facilmente. A entrega f...


In [18]:
# Select 10 reviews for initial AI classification test

sample_reviews = review_ai[
    ["review_id", "review_score", "review_text"]
].head(10).copy()

for i, row in sample_reviews.iterrows():
    print(f"\nReview {i+1}:")
    print("Review score:", row["review_score"])
    print("Review:", row["review_text"])


Review 4:
Review score: 5
Review: Recebi bem antes do prazo estipulado.

Review 5:
Review score: 5
Review: Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa

Review 10:
Review score: 4
Review: aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho

Review 13:
Review score: 4
Review: Mas um pouco ,travando...pelo valor ta Boa.

Review 16:
Review score: 5
Review: Vendedor confiável, produto ok e entrega antes do prazo.

Review 17:
Review score: 2
Review: GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU

Review 20:
Review score: 1
Review: Péssimo

Review 23:
Review score: 5
Review: Loja nota 10

Review 25:
Review score: 5
Review: obrigado pela atençao amim dispensada

Review 28:
Review score: 5
Review: A compra foi realizada facilmente. A entrega foi efetuada muito antes do prazo dado. O pr

In [19]:
pip install openai

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.2 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 2.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 1.6 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.0 MB 2.3 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.5 MB/s eta 0:00:00

   ---- -----------------------------------  1/10 [truststore]
   -------- -------------------------------  2/10 [pydantic-core]
   ------------ ---------------------------  3/10 [jite


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key not found.


In [22]:
import os

print(os.path.exists("../.env"))

False


In [23]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key not found.


In [24]:
import os

print("Current folder:")
print(os.getcwd())

print("\nParent folder contents:")
print(os.listdir(".."))

Current folder:
c:\Users\apran\OneDrive\Desktop\Olist_Customer_Experience_Analytics\notebooks

Parent folder contents:
['data', 'notebooks', 'olist_analytics.db']


In [25]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key not found.


In [26]:
from dotenv import dotenv_values

config = dotenv_values("../.env")

print("Variables found:", list(config.keys()))

Variables found: []


In [27]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully.")
else:
    print("API key not found.")

API key loaded successfully.
